In [1]:
# ============================================================
# M1 — RECORD 114 CORRECTION
# ACCOUNT A
# CELL 1: SETUP
# ============================================================

import os
import re
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)

# The two groups of 201 signal values
MLII_COLUMNS = [f"MLII_{i}" for i in range(201)]
LEAD2_COLUMNS = [f"Lead2_{i}" for i in range(201)]

WAVEFORM_COLUMNS = MLII_COLUMNS + LEAD2_COLUMNS

assert len(MLII_COLUMNS) == 201
assert len(LEAD2_COLUMNS) == 201
assert len(WAVEFORM_COLUMNS) == 402

RECORD_TO_CORRECT = 114

print("=" * 65)
print("RECORD 114 DATASET CORRECTION — ACCOUNT A")
print("=" * 65)

print("MLII columns:", len(MLII_COLUMNS))
print("Lead2 columns:", len(LEAD2_COLUMNS))
print("Total waveform columns:", len(WAVEFORM_COLUMNS))
print("Record to correct:", RECORD_TO_CORRECT)

print("\nSetup complete. No data has been changed.")

RECORD 114 DATASET CORRECTION — ACCOUNT A
MLII columns: 201
Lead2 columns: 201
Total waveform columns: 402
Record to correct: 114

Setup complete. No data has been changed.


In [2]:
# ============================================================
# CELL 2: LOCATE AND LOAD ORIGINAL PROCESSED DATASET
# ============================================================

INPUT_ROOT = "/kaggle/input"

def clean_column_name(name):
    return str(name).strip().strip('"').strip("'").strip()


# ------------------------------------------------------------
# 1. Find the original processed dataset
# ------------------------------------------------------------

processed_candidates = []

for root, dirs, files in os.walk(INPUT_ROOT):

    for filename in files:

        if filename.lower() != "mitbih_mlii_duallead_dataset.csv":
            continue

        full_path = os.path.join(root, filename)

        try:
            header = pd.read_csv(full_path, nrows=0)

            cleaned_headers = {
                clean_column_name(c)
                for c in header.columns
            }

            required_columns = {
                "Record_ID",
                "Label",
                "MLII_0",
                "MLII_200",
                "Lead2_0",
                "Lead2_200"
            }

            if required_columns.issubset(cleaned_headers):
                processed_candidates.append(full_path)

        except Exception as e:
            print("Could not inspect:", full_path, "|", e)


print("Processed dataset candidates:")
for path in processed_candidates:
    print(path)

if len(processed_candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one original processed dataset, "
        f"but found {len(processed_candidates)}. "
        "Stop and inspect the paths."
    )

original_dataset_path = processed_candidates[0]

print("\nSelected original dataset:")
print(original_dataset_path)


# ------------------------------------------------------------
# 2. Load the original dataset
# ------------------------------------------------------------

original_df = pd.read_csv(original_dataset_path)

original_df.columns = [
    clean_column_name(c)
    for c in original_df.columns
]

original_df["Record_ID"] = pd.to_numeric(
    original_df["Record_ID"],
    errors="raise"
).astype(int)

print("\nORIGINAL DATASET SUMMARY")
print("Shape:", original_df.shape)
print("Number of recordings:", original_df["Record_ID"].nunique())
print("Number of waveform columns:", len(WAVEFORM_COLUMNS))


# ------------------------------------------------------------
# 3. Validate expected structure
# ------------------------------------------------------------

missing_columns = [
    c for c in WAVEFORM_COLUMNS
    if c not in original_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing waveform columns: {missing_columns}"
    )

if "Label" not in original_df.columns:
    raise ValueError("Label column is missing.")

if original_df.shape != (106870, 404):
    raise ValueError(
        f"Unexpected dataset shape: {original_df.shape}. "
        "Expected (106870, 404). Stop for inspection."
    )

if original_df["Record_ID"].nunique() != 46:
    raise ValueError("Expected 46 recordings.")

if RECORD_TO_CORRECT not in set(original_df["Record_ID"]):
    raise ValueError("Record 114 was not found.")


# ------------------------------------------------------------
# 4. Inspect record 114
# ------------------------------------------------------------

record114_original = original_df[
    original_df["Record_ID"] == RECORD_TO_CORRECT
].copy()

print("\nRECORD 114 SUMMARY")
print("Total rows:", len(record114_original))

print("\nOriginal annotation symbol counts:")
display(
    record114_original["Label"]
    .value_counts(dropna=False)
    .to_frame("Count")
)


# ------------------------------------------------------------
# 5. Verify valid AAMI beat count
# ------------------------------------------------------------

AAMI_MAP = {
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    "A": "S", "a": "S", "J": "S", "S": "S",
    "V": "V", "E": "V",
    "F": "F",
    "/": "Q", "f": "Q", "Q": "Q"
}

record114_aami = (
    record114_original["Label"]
    .astype(str)
    .str.strip()
    .map(AAMI_MAP)
)

print("\nRecord 114 valid AAMI beat count:",
      int(record114_aami.notna().sum()))

print("\nRecord 114 AAMI class counts:")
display(
    record114_aami
    .value_counts()
    .reindex(["N", "S", "V", "F", "Q"], fill_value=0)
    .to_frame("Count")
)


# ------------------------------------------------------------
# 6. Save basic original-data reference information
# ------------------------------------------------------------

original_shape = original_df.shape
original_record_ids = sorted(
    original_df["Record_ID"].unique().tolist()
)

original_row_count = len(original_df)
original_record114_row_count = len(record114_original)

print("\n" + "=" * 65)
print("CELL 2 VALIDATION")
print("=" * 65)

print("Dataset dimensions correct:", original_shape == (106870, 404))
print("46 recordings present:", len(original_record_ids) == 46)
print("Record 114 rows:", original_record114_row_count)
print("Valid AAMI beats in record 114:",
      int(record114_aami.notna().sum()))

print("\nNo correction has been performed yet.")

Processed dataset candidates:
/kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/01_Processed_Dataset/MITBIH_MLII_DualLead_Dataset.csv

Selected original dataset:
/kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-inputs/ECG_Project_Inputs/01_Processed_Dataset/MITBIH_MLII_DualLead_Dataset.csv

ORIGINAL DATASET SUMMARY
Shape: (106870, 404)
Number of recordings: 46
Number of waveform columns: 402

RECORD 114 SUMMARY
Total rows: 1887

Original annotation symbol counts:


,Count
Label,
N,1820
V,43
A,10
~,7
F,4
J,2
|,1



Record 114 valid AAMI beat count: 1879

Record 114 AAMI class counts:


,Count
Label,
N,1820
S,12
V,43
F,4
Q,0



CELL 2 VALIDATION
Dataset dimensions correct: True
46 recordings present: True
Record 114 rows: 1887
Valid AAMI beats in record 114: 1879

No correction has been performed yet.


In [3]:
# ============================================================
# M1 — RECORD 114 LEAD CORRECTION
# CELL 3: CREATE CORRECTED DATASET
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# 1. Preserve the original dataset
# ------------------------------------------------------------

corrected_df = original_df.copy(deep=True)

print("Original dataset shape:", original_df.shape)
print("Correction will be applied only to Record_ID 114.")


# ------------------------------------------------------------
# 2. Identify all rows belonging to record 114
# ------------------------------------------------------------

record_mask = (
    corrected_df["Record_ID"] == RECORD_TO_CORRECT
)

record114_before = corrected_df.loc[
    record_mask, WAVEFORM_COLUMNS
].copy()

print("\nRecord 114 rows:", int(record_mask.sum()))

if record_mask.sum() != 1887:
    raise ValueError(
        "Unexpected Record 114 row count. "
        "Expected 1887. Correction stopped."
    )


# ------------------------------------------------------------
# 3. Preserve both original lead groups before swapping
# ------------------------------------------------------------

old_mlii_values = corrected_df.loc[
    record_mask, MLII_COLUMNS
].to_numpy(copy=True)

old_lead2_values = corrected_df.loc[
    record_mask, LEAD2_COLUMNS
].to_numpy(copy=True)


# ------------------------------------------------------------
# 4. Swap the two lead groups for Record 114 only
# ------------------------------------------------------------

corrected_df.loc[
    record_mask, MLII_COLUMNS
] = old_lead2_values

corrected_df.loc[
    record_mask, LEAD2_COLUMNS
] = old_mlii_values


print("\nRecord 114 lead groups exchanged.")


# ------------------------------------------------------------
# 5. Check that the correction was performed correctly
# ------------------------------------------------------------

new_mlii_values = corrected_df.loc[
    record_mask, MLII_COLUMNS
].to_numpy()

new_lead2_values = corrected_df.loc[
    record_mask, LEAD2_COLUMNS
].to_numpy()

mlii_swap_correct = np.array_equal(
    new_mlii_values,
    old_lead2_values
)

lead2_swap_correct = np.array_equal(
    new_lead2_values,
    old_mlii_values
)

print("\nSWAP VERIFICATION")
print("MLII now contains previous Lead2 values:",
      mlii_swap_correct)

print("Lead2 now contains previous MLII values:",
      lead2_swap_correct)


# ------------------------------------------------------------
# 6. Verify that other recordings are unchanged
# ------------------------------------------------------------

other_records_mask = ~record_mask

other_records_unchanged = (
    original_df.loc[other_records_mask].reset_index(drop=True)
    .equals(
        corrected_df.loc[other_records_mask].reset_index(drop=True)
    )
)

print("\nOTHER RECORDINGS UNCHANGED:",
      other_records_unchanged)


# ------------------------------------------------------------
# 7. Verify labels, IDs, columns and row count
# ------------------------------------------------------------

labels_unchanged = original_df["Label"].equals(
    corrected_df["Label"]
)

record_ids_unchanged = original_df["Record_ID"].equals(
    corrected_df["Record_ID"]
)

columns_unchanged = (
    original_df.columns.tolist()
    == corrected_df.columns.tolist()
)

shape_unchanged = (
    original_df.shape == corrected_df.shape
)

print("\nSTRUCTURAL VALIDATION")
print("Labels unchanged:", labels_unchanged)
print("Record IDs unchanged:", record_ids_unchanged)
print("Columns unchanged:", columns_unchanged)
print("Dataset dimensions unchanged:", shape_unchanged)


# ------------------------------------------------------------
# 8. Final correction gate
# ------------------------------------------------------------

correction_checks = {
    "Correct_Record114_Row_Count": record_mask.sum() == 1887,
    "MLII_Lead_Swap_Correct": mlii_swap_correct,
    "Lead2_Lead_Swap_Correct": lead2_swap_correct,
    "Other_45_Recordings_Unchanged": other_records_unchanged,
    "Labels_Unchanged": labels_unchanged,
    "Record_IDs_Unchanged": record_ids_unchanged,
    "Columns_Unchanged": columns_unchanged,
    "Dataset_Shape_Unchanged": shape_unchanged
}

print("\n" + "=" * 65)
print("CORRECTION VALIDATION")
print("=" * 65)

for name, passed in correction_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

CORRECTION_PASSED = all(correction_checks.values())

if not CORRECTION_PASSED:
    raise RuntimeError(
        "Correction validation failed. "
        "Corrected output will not be saved."
    )


# ------------------------------------------------------------
# 9. Save corrected dataset separately
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/kaggle/working/ECG_Project/03_Corrected_Dataset"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

corrected_dataset_path = (
    OUTPUT_DIR /
    "MITBIH_MLII_DualLead_Dataset_corrected.csv"
)

corrected_df.to_csv(
    corrected_dataset_path,
    index=False,
    float_format="%.17g"
)

print("\n" + "=" * 65)
print("CORRECTED DATASET SAVED")
print("=" * 65)

print("Output:", corrected_dataset_path)
print("File size (MB):",
      round(corrected_dataset_path.stat().st_size / (1024**2), 2))

print("\nOriginal input dataset was not modified.")
print("No model was trained.")
print("No M0 splits were changed.")

Original dataset shape: (106870, 404)
Correction will be applied only to Record_ID 114.

Record 114 rows: 1887

Record 114 lead groups exchanged.

SWAP VERIFICATION
MLII now contains previous Lead2 values: True
Lead2 now contains previous MLII values: True

OTHER RECORDINGS UNCHANGED: True

STRUCTURAL VALIDATION
Labels unchanged: True
Record IDs unchanged: True
Columns unchanged: True
Dataset dimensions unchanged: True

CORRECTION VALIDATION
Correct_Record114_Row_Count: PASS
MLII_Lead_Swap_Correct: PASS
Lead2_Lead_Swap_Correct: PASS
Other_45_Recordings_Unchanged: PASS
Labels_Unchanged: PASS
Record_IDs_Unchanged: PASS
Columns_Unchanged: PASS
Dataset_Shape_Unchanged: PASS

CORRECTED DATASET SAVED
Output: /kaggle/working/ECG_Project/03_Corrected_Dataset/MITBIH_MLII_DualLead_Dataset_corrected.csv
File size (MB): 181.89

Original input dataset was not modified.
No model was trained.
No M0 splits were changed.


In [4]:
# ============================================================
# CELL 4 — VERIFY SAVED CORRECTED DATASET
# ============================================================

# ------------------------------------------------------------
# 1. Reload the saved corrected CSV
# ------------------------------------------------------------

saved_df = pd.read_csv(corrected_dataset_path)

saved_df.columns = [
    clean_column_name(c)
    for c in saved_df.columns
]

saved_df["Record_ID"] = pd.to_numeric(
    saved_df["Record_ID"],
    errors="raise"
).astype(int)

print("Saved dataset shape:", saved_df.shape)


# ------------------------------------------------------------
# 2. Verify dataset dimensions and column order
# ------------------------------------------------------------

shape_match = saved_df.shape == original_df.shape

column_match = (
    saved_df.columns.tolist()
    == original_df.columns.tolist()
)

print("\nDATASET STRUCTURE")
print("Shape matches original:", shape_match)
print("Column order matches original:", column_match)


# ------------------------------------------------------------
# 3. Verify the corrected record 114
# ------------------------------------------------------------

original_114 = original_df[
    original_df["Record_ID"] == 114
]

saved_114 = saved_df[
    saved_df["Record_ID"] == 114
]

mlii_correct = np.array_equal(
    saved_114[MLII_COLUMNS].to_numpy(),
    original_114[LEAD2_COLUMNS].to_numpy()
)

lead2_correct = np.array_equal(
    saved_114[LEAD2_COLUMNS].to_numpy(),
    original_114[MLII_COLUMNS].to_numpy()
)

print("\nRECORD 114")
print("Rows:", len(saved_114))
print("MLII values correctly swapped:", mlii_correct)
print("Lead2 values correctly swapped:", lead2_correct)


# ------------------------------------------------------------
# 4. Verify all other recordings remain unchanged
# ------------------------------------------------------------

other_mask = saved_df["Record_ID"] != 114

other_records_match = (
    saved_df.loc[other_mask]
    .reset_index(drop=True)
    .equals(
        original_df.loc[other_mask]
        .reset_index(drop=True)
    )
)

print("\nOTHER RECORDINGS")
print("All other recordings unchanged:", other_records_match)


# ------------------------------------------------------------
# 5. Verify labels and record IDs
# ------------------------------------------------------------

labels_match = (
    saved_df["Label"].equals(original_df["Label"])
)

record_ids_match = (
    saved_df["Record_ID"].equals(original_df["Record_ID"])
)

print("\nMETADATA")
print("Labels unchanged:", labels_match)
print("Record IDs unchanged:", record_ids_match)


# ------------------------------------------------------------
# 6. Final saved-file validation
# ------------------------------------------------------------

saved_file_checks = {
    "Saved_shape_correct": shape_match,
    "Column_order_correct": column_match,
    "Record114_MLII_correct": mlii_correct,
    "Record114_Lead2_correct": lead2_correct,
    "Other_records_unchanged": other_records_match,
    "Labels_unchanged": labels_match,
    "Record_IDs_unchanged": record_ids_match
}

print("\n" + "=" * 65)
print("SAVED DATASET VALIDATION")
print("=" * 65)

for name, passed in saved_file_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

SAVED_DATASET_PASSED = all(saved_file_checks.values())

print("\nFINAL STATUS:",
      "PASS" if SAVED_DATASET_PASSED else "FAIL")

if not SAVED_DATASET_PASSED:
    raise RuntimeError(
        "Saved dataset validation failed. Stop here."
    )

print("\nSaved corrected dataset verified successfully.")

Saved dataset shape: (106870, 404)

DATASET STRUCTURE
Shape matches original: True
Column order matches original: True

RECORD 114
Rows: 1887
MLII values correctly swapped: True
Lead2 values correctly swapped: True

OTHER RECORDINGS
All other recordings unchanged: True

METADATA
Labels unchanged: True
Record IDs unchanged: True

SAVED DATASET VALIDATION
Saved_shape_correct: PASS
Column_order_correct: PASS
Record114_MLII_correct: PASS
Record114_Lead2_correct: PASS
Other_records_unchanged: PASS
Labels_unchanged: PASS
Record_IDs_unchanged: PASS

FINAL STATUS: PASS

Saved corrected dataset verified successfully.
